# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{getattr(metadata, 'name', 'No dataset name')}: {getattr(metadata, 'description', 'No dataset description')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Get all record sets (by @id) from the Croissant schema
record_sets = dataset.metadata.record_sets

print('Available Record Sets:')
for rs in record_sets:
    print(f"- @id: {rs.id} | name: {getattr(rs, 'name', 'N/A')}")
    print('    Fields (by @id):')
    for field in getattr(rs, 'fields', []):
        print(f"        - @id: {field.id} | name: {getattr(field, 'name', 'N/A')} (type: {getattr(field, 'data_type', 'N/A')})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# List all record set @ids
record_set_ids = [rs.id for rs in record_sets]
print('All record set @ids:', record_set_ids)

# Extract data for all record sets into pandas DataFrames
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Extracted DataFrame with shape {df.shape} for record set {record_set_id}")
        print("Available columns (field @id):", df.columns.tolist())
        print(df.head())
    else:
        print(f"No records found for record set {record_set_id}")
    print()
# Let's select the main survey records set for further analysis (usually the largest record set). 
main_record_set_id = None
max_rows = 0
for k, v in dataframes.items():
    if len(v) > max_rows:
        main_record_set_id = k
        max_rows = len(v)
print(f"Selected main record set: {main_record_set_id} (with {max_rows} rows)")
print(dataframes[main_record_set_id].head())
main_df = dataframes[main_record_set_id]

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example: Filter, normalize and group on survey score fields (e.g., GAD-7, PHQ-9, PSQ)
# First, try to find field @ids for these survey fields

expected_numeric_fields = ['gad7_total_score', 'phq9_total_score', 'psq_score']
available_fields = main_df.columns.tolist()

# Attempt automatic matching for numeric field @ids
numeric_field_id = None
for fld in available_fields:
    if any(expected in fld.lower() for expected in expected_numeric_fields):
        numeric_field_id = fld
        break
if numeric_field_id is None:
    # Fallback: use the first float/integer-like field
    for fld in available_fields:
        if pd.api.types.is_numeric_dtype(main_df[fld]):
            numeric_field_id = fld
            break
print(f"Using field for numeric analysis: {numeric_field_id}")

# Determine a threshold for filtering (e.g., > 10 for survey scores)
threshold = 10
filtered_df = main_df[main_df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold} (n={len(filtered_df)}):")
print(filtered_df.head())

# Normalizing the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / (filtered_df[numeric_field_id].std())
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by an attribute, e.g, gender, education, or village
possible_group_fields = ['gender', 'village', 'level_of_education']
group_field = None
for fld in possible_group_fields:
    for col in available_fields:
        if fld in col.lower():
            group_field = col
            break
    if group_field is not None:
        break
if group_field is not None:
    print(f'Grouping by: {group_field}')
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(grouped_df)
else:
    print('No suitable group field found.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(main_df[numeric_field_id].dropna(), bins=15, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

if group_field is not None:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=group_field, y=numeric_field_id, data=main_df)
    plt.title(f"{numeric_field_id} Scores by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load, explore, and analyze a Croissant-FAIR dataset using `mlcroissant`. We examined available record sets and their fields (referencing them by `@id`), loaded the main survey records into a DataFrame, and performed basic EDA including filtering, normalization, and visualizations. Further domain-specific analysis can be carried out using the extracted DataFrames, always referencing data elements by their Croissant `@id`s for full reproducibility and interoperability.